# 05 — GSEA Pathway Over-Representation

**Section:** Evaluation layer (PROMETHEE II, CART baseline, statistical testing, GSEA)
**Depends on:** `02_synthetic_rule_pools.ipynb`, `03_promethee_ranking.ipynb`
**Feeds into:** report write-up (biological plausibility check on top-ranked rules)

## Recommended reading before this notebook

- **Subramanian et al., "Gene set enrichment analysis..."** — the GSEA method this notebook's
  over-representation test is a (simpler, hypergeometric) relative of.
- **Fang et al., "GSEApy..."** — the Python package used here.
- **Methodology §4.6.3** — GSEApy over-representation of each rule pool's `genes` against
  KEGG/Enrichr, reporting enrichment p-values.

## A network constraint, addressed up front

The real pipeline calls `gseapy.enrichr(gene_list=..., gene_sets='KEGG_2021_Human', ...)`,
which queries Enrichr's public API over the network. **This sandbox's network allow-list
doesn't include Enrichr**, and the synthetic `genes` values here (`GENE01`, `PATHWAY_19`, ...)
aren't real gene symbols KEGG would recognise anyway. So this notebook exercises the identical
statistical machinery — `gseapy.enrich()`, the same hypergeometric over-representation test
`enrichr()` uses internally — against a **locally-defined synthetic gene-set library** built
from the same planted ground truth notebook 01/02 already use, rather than a live KEGG query.
This proves the code path and the statistic before real gene symbols and real network access
exist; when Angelique's real `feature_map.json` lands with real driver-gene symbols, only the
gene-set source changes (`gene_sets='KEGG_2021_Human'` instead of the synthetic dict below),
not the surrounding logic.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import gseapy as gp

sys.path.insert(0, "..")  # repo root, so `import src...` resolves

FIXTURE_DIR = Path("../tests/fixtures/synthetic")
REPORT_DIR = FIXTURE_DIR / "reports"

feature_map = json.load(open(FIXTURE_DIR / "feature_map.json"))
ground_truth = json.load(open(FIXTURE_DIR / "synthetic_ground_truth_meta.json"))

from src.common.schema import load_rule_pool
POOL_NAMES = ["ga_baseline", "ga_sam", "nsga_baseline", "nsga_sam"]
pools = {name: load_rule_pool(FIXTURE_DIR / "rules" / f"{name}.json", feature_map=feature_map) for name in POOL_NAMES}
rule_lookup = {r["id"]: r for pool in pools.values() for r in pool["rules"]}

combined = pd.read_csv(REPORT_DIR / "promethee_ranked_rules.csv")
print(f"loaded {len(combined)} ranked rules, {len(rule_lookup)} rule definitions")


loaded 60 ranked rules, 60 rule definitions


## 1. Building a synthetic gene-set library

Real KEGG/Enrichr gene sets are named pathways (e.g. `"Apoptosis - Homo sapiens"`) with lists
of member gene symbols. To stand in for that structure: one `SIGNAL_FAMILY` set built from the
planted `TRUE_SIGNAL_GENES`/`TRUE_SIGNAL_PATHWAYS` (resolved through `feature_map.json` to
their `gene_symbol` form, matching how real rule `genes` fields are resolved) plus a few
unrelated decoys for realism, and three `NOISE_FAMILY_*` sets of equal size drawn from the
remaining, unrelated features. If the over-representation test correctly flags `SIGNAL_FAMILY`
as most enriched among top-ranked rules' genes, that's a genuine (if small-N) validation that
the statistic is wired correctly — this is the same kind of "recovers the planted signal"
check notebooks 02 and 03 already ran, extended to the GSEA step.


In [2]:
all_symbols = sorted({v["gene_symbol"] for v in feature_map.values()})

true_signal_features = set(ground_truth["true_signal_genes"].keys()) | set(ground_truth["true_signal_pathways"].keys())
resolved_true = {v["gene_symbol"] for f, v in feature_map.items() if f in true_signal_features}
print(f"planted signal resolves to {len(resolved_true)} symbols: {sorted(resolved_true)}")

rng = np.random.default_rng(7)
remaining = [s for s in all_symbols if s not in resolved_true]
rng.shuffle(remaining)

SET_SIZE = 8  # rough KEGG-pathway-sized gene set, arbitrary but consistent across families
synthetic_gene_sets = {"SIGNAL_FAMILY": sorted(resolved_true | set(remaining[: SET_SIZE - len(resolved_true)]))}
decoy_pool = remaining[SET_SIZE - len(resolved_true):]
n_noise_sets = 3
for i in range(n_noise_sets):
    chunk = decoy_pool[i * SET_SIZE: (i + 1) * SET_SIZE]
    synthetic_gene_sets[f"NOISE_FAMILY_{i + 1}"] = sorted(chunk)

for name, members in synthetic_gene_sets.items():
    print(f"{name}: {len(members)} members")


planted signal resolves to 7 symbols: ['GENE06', 'GENE11', 'GENE12', 'GENE17', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
SIGNAL_FAMILY: 8 members
NOISE_FAMILY_1: 8 members
NOISE_FAMILY_2: 8 members
NOISE_FAMILY_3: 8 members


## 2. Query gene list: union of top-ranked rules' genes

Per condition (`extractor`/`condition` pair), the query gene list is the union of `genes`
across that group's top-10 φ-ranked rules from notebook 03. Background is the full 58-feature
universe (`feature_map.json`), matching how real KEGG background would be "all genes measured
in this experiment" rather than the whole genome (`gseapy.enrich`'s `background` parameter).


In [3]:
top10 = (
    combined.sort_values("phi", ascending=False)
    .groupby(["extractor", "condition"])
    .head(10)
)

def query_genes_for_group(group_df: pd.DataFrame) -> list[str]:
    genes = set()
    for rid in group_df["id"]:
        genes.update(rule_lookup[rid]["genes"])
    return sorted(genes)

query_gene_lists = {
    f"{extractor}_{condition}": query_genes_for_group(group)
    for (extractor, condition), group in top10.groupby(["extractor", "condition"])
}
{k: len(v) for k, v in query_gene_lists.items()}


{'ga_baseline': 15, 'ga_sam': 19, 'nsga_baseline': 13, 'nsga_sam': 15}

## 3. Run over-representation per group

`gseapy.enrich()` runs the same hypergeometric test `enrichr()` does, just against a supplied
`gene_sets` dict instead of a live Enrichr query — no network call, exactly the offline
substitution described in the intro above.


In [4]:
enrichment_results = {}
for group_name, gene_list in query_gene_lists.items():
    res = gp.enrich(gene_list=gene_list, gene_sets=synthetic_gene_sets, background=all_symbols, outdir=None)
    df = res.results.copy()
    df.insert(0, "group", group_name)
    enrichment_results[group_name] = df
    print(f"--- {group_name} (query n={len(gene_list)}) ---")
    print(df[["Term", "Overlap", "P-value", "Adjusted P-value"]].to_string(index=False))
    print()

all_enrichment = pd.concat(enrichment_results.values(), ignore_index=True)


--- ga_baseline (query n=15) ---
          Term Overlap  P-value  Adjusted P-value
NOISE_FAMILY_2     2/8 0.672177          0.672177
 SIGNAL_FAMILY     4/8 0.109723          0.219447

--- ga_sam (query n=19) ---
          Term Overlap  P-value  Adjusted P-value
NOISE_FAMILY_1     1/8 0.967903          0.967903
NOISE_FAMILY_2     2/8 0.815441          0.967903
NOISE_FAMILY_3     5/8 0.066994          0.267976
 SIGNAL_FAMILY     3/8 0.524378          0.967903

--- nsga_baseline (query n=13) ---
          Term Overlap  P-value  Adjusted P-value
NOISE_FAMILY_1     4/8 0.066033          0.198098
NOISE_FAMILY_2     1/8 0.887545          0.887545
 SIGNAL_FAMILY     3/8 0.248328          0.372492

--- nsga_sam (query n=15) ---
          Term Overlap  P-value  Adjusted P-value
NOISE_FAMILY_1     4/8 0.109723          0.219447
NOISE_FAMILY_2     1/8 0.924349          0.924349
NOISE_FAMILY_3     3/8 0.338220          0.450960
 SIGNAL_FAMILY     4/8 0.109723          0.219447



## 4. Sanity check

Does `SIGNAL_FAMILY` come out as the lowest (most significant) p-value in each group, ahead of
the three noise families? This won't be a clean sweep at this sample size (small gene lists,
small gene-set universe → limited statistical power), but a majority pointing the right way is
the signal to look for.


In [5]:
summary_rows = []
for group_name, df in enrichment_results.items():
    ranked = df.sort_values("P-value")
    top_term = ranked.iloc[0]["Term"]
    signal_rank = int((ranked["Term"] == "SIGNAL_FAMILY").to_numpy().nonzero()[0][0]) + 1
    signal_p = float(ranked.loc[ranked["Term"] == "SIGNAL_FAMILY", "P-value"].iloc[0])
    summary_rows.append({
        "group": group_name, "top_term": top_term,
        "signal_family_rank": signal_rank, "signal_family_p": signal_p,
    })

summary_df = pd.DataFrame(summary_rows)
n_correct = int((summary_df["signal_family_rank"] == 1).sum())
print(f"SIGNAL_FAMILY ranked #1 by p-value in {n_correct}/{len(summary_df)} groups")
summary_df


SIGNAL_FAMILY ranked #1 by p-value in 1/4 groups


,group,top_term,signal_family_rank,signal_family_p
0,ga_baseline,SIGNAL_FAMILY,1,0.109723
1,ga_sam,NOISE_FAMILY_3,2,0.524378
2,nsga_baseline,NOISE_FAMILY_1,2,0.248328
3,nsga_sam,NOISE_FAMILY_1,2,0.109723


## 5. Save outputs

Same placement convention as notebooks 03/04 — synthetic-input-derived output stays under
`tests/fixtures/synthetic/reports/`.


In [6]:
all_enrichment.to_csv(REPORT_DIR / "gsea_enrichment_results.csv", index=False)
summary_df.to_csv(REPORT_DIR / "gsea_sanity_check.csv", index=False)
print(f"wrote enrichment results and sanity-check summary -> {REPORT_DIR}")


wrote enrichment results and sanity-check summary -> ../tests/fixtures/synthetic/reports


## Known gaps, to address when real data arrives

- **Not real KEGG/Enrichr** — this notebook proves the statistical/code path only. Once
  Angelique's real `feature_map.json` (real driver-gene symbols) exists, swap
  `synthetic_gene_sets` for `gene_sets="KEGG_2021_Human"` (or similar) and switch
  `gp.enrich()` → `gp.enrichr()`, which needs network access to Enrichr not currently
  available in this sandbox — confirm that's available wherever this actually runs for real
  (NICOG HPC or a team member's machine).
- **Background universe is only 58 features** (this synthetic dataset's full feature count),
  nowhere near a real ~572-feature (534 driver genes + 38 pathways) or genome-wide background
  — real p-values will look very different in scale, though the *logic* being tested here
  (does the top-ranked rules' gene set land disproportionately in the "true" family) carries
  over directly.
- Per-drug repetition, same as notebooks 03/04, once the drug shortlist is fixed.

## Next up

- **`06_cart_baseline.ipynb`** — CART surrogate rules, scored with the same shared
  `metrics.py` functions and folded into notebook 03's unified PROMETHEE run for an
  evolutionary-vs-CART comparison.
